In [1]:
from IPython.display import display, Markdown
import numpy as np
from scipy.stats import betabinom

### Hela cell residence time


In [2]:
# Omega range in minutes (Synthesis + Residence time)
omega_min_minutes = 0.29
omega_max_minutes = 0.65

# Polymerization rate (nucleotides per second)
pol_gamma_rate = 37 

# --- Biological Constant (Implicit in the calculation) ---
# Standard length of human mitochondrial 7S DNA (D-loop strand)
# Typically  ~650nt. 
# (Note: 0.29 min * 60 sec/min * 37 nt/sec = ~644 nt, confirming ~650 is the basis)
len_7S_dna = 650 # nucleotides

# --- Calculation ---

# 1. Calculate time required strictly for synthesis (t = length / rate)
t_synthesis_seconds = len_7S_dna / pol_gamma_rate

# 2. Convert omega range to seconds
omega_min_seconds = omega_min_minutes * 60
omega_max_seconds = omega_max_minutes * 60

# 3. Derive Residence Time (Residence = Total Time - Synthesis Time)
# Residence time cannot be negative, so we take max(0, val) for the lower bound
residence_time_min = max(0, omega_min_seconds - t_synthesis_seconds)
residence_time_max = omega_max_seconds - t_synthesis_seconds

In [3]:
display(Markdown(f"""
### Parameters

| Parameter | Value |
|----------|-------|
| Pol Gamma Rate | {pol_gamma_rate} nt/s |
| 7S DNA Length | {len_7S_dna} nt |
| Synthesis Time | {t_synthesis_seconds:.2f} seconds |
| Omega Range (min) | [{omega_min_minutes}, {omega_max_minutes}] |
| Omega Range (sec) | [{omega_min_seconds:.2f}, {omega_max_seconds:.2f}] |

### Calculated Residence Time

| Bound | Time (seconds) |
|------|----------------|
| Lower Bound | {residence_time_min:.2f} |
| Upper Bound | {residence_time_max:.2f} |
"""))



### Parameters

| Parameter | Value |
|----------|-------|
| Pol Gamma Rate | 37 nt/s |
| 7S DNA Length | 650 nt |
| Synthesis Time | 17.57 seconds |
| Omega Range (min) | [0.29, 0.65] |
| Omega Range (sec) | [17.40, 39.00] |

### Calculated Residence Time

| Bound | Time (seconds) |
|------|----------------|
| Lower Bound | 0.00 |
| Upper Bound | 21.43 |


### Nucleoid Pool size correlation

In [4]:


# --- 1. Minimal model of nucleoid ---
class Nucleoid:
    def __init__(self, pool_size):
        self.pool_size = pool_size

# Distribution parameters
NUC_DIST_PARAMS = {
    'right_102': [10, 2], # The default used in the notebook (High mean)
    'normal': [2, 2], # The default used in the notebook (High mean)
    'left': [2, 5],       # A left-skewed example (Low mean)
}

def generate_initial_pools(dist_name, num_nucleoids, max_pool=165690):
    """Simulates the initialization step of the simulation."""
    if dist_name not in NUC_DIST_PARAMS: return []
    alpha, beta = NUC_DIST_PARAMS[dist_name]
    
    # Random State for reproducibility
    rng = np.random.default_rng(42)
    
    # Generate pools exactly as the simulation does (Beta-Binomial)
    pool_sizes = betabinom.rvs(max_pool, alpha, beta, size=num_nucleoids, random_state=rng)
    return pool_sizes

# Constants from text
N_nucleoids = 490 

# Right skew distribution ('right_102')
pools_right102 = generate_initial_pools('right_102', N_nucleoids)
total_right102 = np.sum(pools_right102)
pools_left = generate_initial_pools('left', N_nucleoids)
total_left = np.sum(pools_left)
pools_normal = generate_initial_pools('normal', N_nucleoids)
total_normal = np.sum(pools_normal)

display(Markdown(f"""
### Nucleotide Pool Statistics N={N_nucleoids}

| Condition | Sum of dNTPs | Mean Filling |
|----------|---------------|--------------|
| Right skewed | {total_right102:.2e} | {np.mean(pools_right102)/165690:.1%} |
| Normal | {total_normal:.2e} | {np.mean(pools_normal)/165690:.1%} |
| Left skewed | {total_left:.2e} | {np.mean(pools_left)/165690:.1%} |
"""))



### Nucleotide Pool Statistics N=490

| Condition | Sum of dNTPs | Mean Filling |
|----------|---------------|--------------|
| Right skewed | 6.73e+07 | 82.9% |
| Normal | 4.12e+07 | 50.8% |
| Left skewed | 2.41e+07 | 29.7% |
